# E-Commerce Recommendation System — Analysis Notebook

This notebook walks through the full Business Analytics pipeline for the
**Recommendation System Prototype for an E-Commerce Platform Using User–Item Interactions**:

1. Load & inspect the raw dataset
2. Clean & preprocess the data
3. Exploratory Data Analysis (Descriptive & Diagnostic Analytics)
4. Build the Item-Based Collaborative Filtering recommender (Predictive Analytics)
5. Generate sample recommendations
6. Evaluate the model (Precision@K, Recall@K, Hit Rate@K)
7. Business insights & recommendations (Prescriptive Analytics)

> Run all cells top-to-bottom. The dataset must be present at `../data/dataset.csv`
> (or any CSV/Excel file placed in the `data/` folder).

In [1]:
# Setup: make the src/ package importable
import sys
from pathlib import Path

SRC_DIR = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
%matplotlib inline

## 1. Load the dataset

In [2]:
from data_loader import load_dataset

raw_df, column_map, error = load_dataset()

if error:
    print(f"[ERROR] {error}")
else:
    print(f"Loaded dataset with shape: {raw_df.shape}")
    print("Detected column mapping:")
    for k, v in column_map.items():
        print(f"  {k:15s} -> {v}")

raw_df.head()

Loaded dataset with shape: (150000, 19)
Detected column mapping:
  user_id         -> user_id
  product_id      -> product_id
  interaction     -> interaction_type
  timestamp       -> timestamp
  category        -> product_category
  brand           -> brand
  price           -> price


,user_id,product_id,timestamp,session_id,interaction_type,device_type,location,price,discount,product_category,brand,user_age,user_gender,loyalty_score,previous_purchase_count,avg_purchase_value,search_keywords,rating,purchase
0,U0861,P1416,2024-06-25 08:32:00,S0940,view,mobile,Los Angeles,292.99,5,Beauty,Apple,46,Male,19,18,195.46,laptop,NaN,0
1,U3773,P1710,2024-01-28 10:19:00,S5346,click,mobile,Miami,147.48,5,Beauty,LG,38,Male,7,2,11.11,makeup kit,5.0,0
2,U3093,P0167,2024-05-22 16:16:00,S8366,add_to_cart,mobile,Los Angeles,400.47,30,Electronics,Puma,39,Male,20,49,261.12,blender,NaN,0
3,U0467,P0402,2024-04-19 14:19:00,S0208,view,tablet,New York,476.52,20,Apparel,LG,28,Female,63,36,24.81,laptop,NaN,0
4,U4427,P0733,2024-08-09 23:05:00,S4254,view,desktop,Chicago,186.98,30,Beauty,Nike,44,Female,22,41,36.16,running shoes,NaN,0


## 2. Clean & preprocess the data

In [3]:
from preprocessing import clean_dataset, get_dataset_stats

df = clean_dataset(raw_df, column_map)
stats = get_dataset_stats(df)
stats

PREPROCESSING SUMMARY
Original shape:          (150000, 19)
Cleaned shape:            (150000, 9)
Total missing (raw):      74879
Invalid user/product rows dropped: 0
Duplicate records found & removed: 0
Unique users:             4999
Unique products:          1999
Total interactions:       150000
Interaction type counts:
interaction
view           75215
click          44989
add_to_cart    22217
purchase        7579


{'n_rows': 150000,
 'n_users': 4999,
 'n_products': 1999,
 'n_interactions': 150000,
 'interaction_counts': {'view': 75215,
  'click': 44989,
  'add_to_cart': 22217,
  'purchase': 7579},
 'has_category': np.True_,
 'has_brand': np.True_,
 'has_price': np.True_,
 'has_timestamp': np.True_}

In [4]:
df.head()

,user_id,product_id,interaction,timestamp,product_name,category,brand,price,interaction_weight
0,U0861,P1416,view,2024-06-25 08:32:00,None,Beauty,Apple,292.99,1
1,U3773,P1710,click,2024-01-28 10:19:00,None,Beauty,LG,147.48,2
2,U3093,P0167,add_to_cart,2024-05-22 16:16:00,None,Electronics,Puma,400.47,3
3,U0467,P0402,view,2024-04-19 14:19:00,None,Apparel,LG,476.52,1
4,U4427,P0733,view,2024-08-09 23:05:00,None,Beauty,Nike,186.98,1


## 3. Exploratory Data Analysis
### Descriptive & Diagnostic Analytics — *What happened, and why?*

In [5]:
from eda import (
    interaction_distribution, top_n_products, top_n_users,
    category_distribution, brand_distribution, price_summary,
    interactions_over_time, purchase_rate,
)

print("Interaction type distribution:")
display(interaction_distribution(df))

print(f"\nOverall purchase rate: {purchase_rate(df)}%")

Interaction type distribution:


,interaction,count
0,view,75215
1,click,44989
2,add_to_cart,22217
3,purchase,7579



Overall purchase rate: 5.05%


In [6]:
fig, ax = plt.subplots(figsize=(7,5))
dist = interaction_distribution(df)
sns.barplot(data=dist, x="interaction", y="count", hue="interaction", palette="viridis", ax=ax, legend=False)
ax.set_title("Interaction Type Distribution")
ax.set_xlabel("Interaction Type")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [7]:
top_products = top_n_products(df, 10)
display(top_products)

fig, ax = plt.subplots(figsize=(8,6))
sns.barplot(data=top_products.sort_values("interaction_count"), x="interaction_count", y="product_id",
            hue="product_id", palette="mako", ax=ax, legend=False)
ax.set_title("Top 10 Products by Interaction Count")
plt.tight_layout()
plt.show()

,product_id,interaction_count
0,P1843,102
1,P1026,100
2,P0729,100
3,P1146,99
4,P1042,99
5,P1252,99
6,P0746,99
7,P0846,99
8,P0354,99
9,P1124,98


In [8]:
top_users = top_n_users(df, 10)
display(top_users)

,user_id,interaction_count
0,U4483,56
1,U3033,52
2,U4840,51
3,U2157,50
4,U1621,50
5,U0816,48
6,U1725,48
7,U2666,48
8,U3332,48
9,U3658,48


In [9]:
cat_dist = category_distribution(df)
if not cat_dist.empty:
    display(cat_dist)
    fig, ax = plt.subplots(figsize=(7,5))
    sns.barplot(data=cat_dist, x="count", y="category", hue="category", palette="crest", ax=ax, legend=False)
    ax.set_title("Category Distribution")
    plt.tight_layout()
    plt.show()
else:
    print("No category column available in this dataset.")

,category,count
0,Apparel,30121
1,Home,30072
2,Beauty,30049
3,Electronics,29990
4,Sports,29768


In [10]:
brand_dist = brand_distribution(df)
if not brand_dist.empty:
    display(brand_dist.head(10))
else:
    print("No brand column available in this dataset.")

,brand,count
0,Apple,21537
1,Nike,21494
2,LG,21448
3,Adidas,21441
4,Samsung,21438
5,Sony,21346
6,Puma,21296


In [11]:
price_stats = price_summary(df)
print("Price summary statistics:")
price_stats

Price summary statistics:


{'mean': 251.68342093333334,
 'median': 251.31,
 'min': 5.0,
 'max': 500.0,
 'std': 142.54753952686352}

In [12]:
trend = interactions_over_time(df, freq="D")
if not trend.empty:
    fig, ax = plt.subplots(figsize=(10,5))
    ax.plot(trend["period"], trend["count"], color="darkorange")
    ax.set_title("Interaction Trends Over Time")
    ax.set_xlabel("Date")
    ax.set_ylabel("Number of Interactions")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("No timestamp column available in this dataset.")

## 4. Build the Recommendation Engine
### Predictive Analytics — *What might the customer want?*

We build a **User × Item interaction matrix** using weighted interactions
(`view=1, click=2, add_to_cart=3, purchase=5`), then compute **item-item
cosine similarity** to power an item-based collaborative filtering model.

In [13]:
from recommender import ItemBasedRecommender

recommender = ItemBasedRecommender(df)
print(f"User-item matrix shape: {recommender.user_item_matrix.shape}")
print(f"Item-item similarity matrix shape: {recommender.item_similarity.shape}")

User-item matrix shape: (4999, 1999)
Item-item similarity matrix shape: (1999, 1999)


## 5. Generate sample recommendations

In [14]:
sample_user = recommender.known_users()[0]
print(f"Sample recommendations for user: {sample_user}\n")

print("Purchase/interaction history:")
display(recommender.user_history(sample_user)[["product_id", "interaction", "category"]].head(10))

print("\nTop-5 recommendations:")
recommender.get_recommendations(sample_user, n=5)

Sample recommendations for user: U0001

Purchase/interaction history:


,product_id,interaction,category
23991,P1443,view,Electronics
40262,P1927,view,Beauty
60063,P0930,click,Sports
131450,P1580,add_to_cart,Home
36415,P1134,click,Beauty
83339,P1602,view,Apparel
110098,P1665,click,Sports
32883,P1699,view,Electronics
101720,P0654,add_to_cart,Home
137479,P0486,view,Home



Top-5 recommendations:


,rank,product_id,product_name,category,score
0,1,P0512,N/A,Sports,1.4029
1,2,P0381,N/A,Beauty,1.3492
2,3,P0741,N/A,Electronics,1.3203
3,4,P0179,N/A,Electronics,1.3189
4,5,P0424,N/A,Sports,1.3186


## 6. Model Evaluation

We use a **leave-one-out** evaluation: for each eligible user, the most
recent interaction is held out as the test item, the model is trained on
the rest, and we check whether the held-out item appears in the user's
top-K recommendations.

In [15]:
from evaluation import run_evaluation

eval_results = run_evaluation(df, ks=(5, 10))
eval_results

,k,precision,recall,hit_rate,n_users_evaluated
0,5,0.0000,0.000,0.000,500
1,10,0.0004,0.004,0.004,500


## 7. Business Insights
### Prescriptive Analytics — *What should the business do?*

In [16]:
interaction_counts = stats["interaction_counts"]
total = stats["n_interactions"]
most_common_interaction = max(interaction_counts, key=interaction_counts.get)
most_interacted_product = df["product_id"].value_counts().idxmax()

print(f"- Most interacted product: {most_interacted_product} "
      f"({df['product_id'].value_counts().max():,} interactions)")
print(f"- Most common interaction type: {most_common_interaction} "
      f"({100*interaction_counts[most_common_interaction]/total:.1f}% of all interactions)")
if df["category"].notna().any():
    top_cat = df["category"].value_counts().idxmax()
    print(f"- Most popular category: {top_cat} "
          f"({100*df['category'].value_counts().max()/total:.1f}% of all interactions)")
if "purchase" in interaction_counts:
    print(f"- Purchase interactions represent "
          f"{100*interaction_counts['purchase']/total:.1f}% of all recorded interactions")
print(f"- Number of active users: {stats['n_users']:,}")

- Most interacted product: P1843 (102 interactions)
- Most common interaction type: view (50.1% of all interactions)
- Most popular category: Apparel (20.1% of all interactions)
- Purchase interactions represent 5.1% of all recorded interactions
- Number of active users: 4,999


**Recommended business actions:**
- Personalize product recommendations for each customer using the collaborative filtering model above.
- Cross-sell related products within a customer's most-engaged categories.
- Prioritize marketing spend around top-performing products/categories.
- Investigate categories/products with high views but low purchase conversion.